In [13]:
import os
from pathlib import Path

# Set up explicit path resolution
data_dir = Path("../data").resolve()
data_dir.mkdir(parents=True, exist_ok=True)
csv_path = data_dir / "SAML-D.csv"

# Point Kaggle API to our local folder for authentication
os.environ['KAGGLE_CONFIG_DIR'] = str(data_dir)

print(f"Data Directory Context: {data_dir}")

# Check if dataset already exists before downloading
if csv_path.exists():
    print(f"✅ Dataset already exists at {csv_path.name}. Skipping download.")
else:
    print("⚠️ Dataset not found locally. Initiating Kaggle download...")
    try:
        import kaggle
        kaggle.api.dataset_download_files(
            'berkanoztas/synthetic-transaction-monitoring-dataset-aml', 
            path=str(data_dir), 
            unzip=True
        )
        print("✅ Download and extraction complete!")
    except Exception as e:
        print(f"❌ Failed to download dataset. Ensure kaggle.json is in {data_dir}.")
        print(f"Error details: {e}")

# Verify file footprint
for file in data_dir.iterdir():
    if file.is_file():
        print(f" - {file.name} ({file.stat().st_size / (1024*1024):.2f} MB)")

Data Directory Context: /media/storage/mlops-governance-reference/project-3-aml-streaming/data
✅ Dataset already exists at SAML-D.csv. Skipping download.
 - kaggle.json (0.00 MB)
 - SAML-D.csv (950.02 MB)


In [14]:
import pandas as pd
import numpy as np

if not csv_path.exists():
    raise FileNotFoundError(f"Cannot proceed: {csv_path} is missing from the data directory.")

print(f"Reading first 500,000 contiguous rows from {csv_path.name}...")

# Robust memory management: explicitly define types and needed columns
cols_to_use = ['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Is_laundering', 'Laundering_type']
dtypes = {'Sender_account': 'str', 'Receiver_account': 'str', 'Amount': 'float32', 'Is_laundering': 'int8'}

# Read contiguous chunk to preserve actual user behavior over time
df = pd.read_csv(csv_path, usecols=cols_to_use, dtype=dtypes, nrows=500000)

# Robust Datetime processing
df['Timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.drop(columns=['Date', 'Time'], inplace=True)

# CRITICAL: Hard-sort chronologically to guarantee time-series integrity
df = df.sort_values(by='Timestamp').reset_index(drop=True)

print(f"✅ Contiguous Slice Loaded! Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Timeline Span: {df['Timestamp'].min()} to {df['Timestamp'].max()}")
print(f"Laundering Transactions in Slice: {df['Is_laundering'].sum()}")
df.head()

Reading first 500,000 contiguous rows from SAML-D.csv...
✅ Contiguous Slice Loaded! Memory Usage: 93.53 MB
Timeline Span: 2022-10-07 10:35:19 to 2022-10-24 17:02:50
Laundering Transactions in Slice: 556


,Sender_account,Receiver_account,Amount,Is_laundering,Laundering_type,Timestamp
0,8724731955,2769355426,1459.150024,0,Normal_Cash_Deposits,2022-10-07 10:35:19
1,1491989064,8401255335,6019.640137,0,Normal_Fan_Out,2022-10-07 10:35:20
2,287305149,4404767002,14328.440430,0,Normal_Small_Fan_Out,2022-10-07 10:35:20
3,5376652437,9600420220,11895.000000,0,Normal_Fan_In,2022-10-07 10:35:21
4,9614186178,3803336972,115.250000,0,Normal_Cash_Deposits,2022-10-07 10:35:21


In [15]:
print("=== CLASS IMBALANCE METRICS ===")
imbalance = df['Is_laundering'].value_counts(normalize=True) * 100
counts = df['Is_laundering'].value_counts().rename(index={0: 'Normal (0)', 1: 'Laundering (1)'})
print(counts)
print(f"Laundering Percentage: {imbalance.get(1, 0):.4f}%\n")

print("=== MONEY LAUNDERING TYPOLOGY BREAKDOWN ===")
# Safely filter and count typologies, ignoring NaNs used for normal transactions
if 'Laundering_type' in df.columns:
    typologies = df[df['Is_laundering'] == 1]['Laundering_type'].value_counts().reset_index()
    typologies.columns = ['Laundering_type', 'Count']
    print(typologies.to_string(index=False) if not typologies.empty else "No labeled typologies in this slice.")

print("\n=== BEHAVIORAL FREQUENCY PER SENDER ACCOUNT ===")
sender_counts = df.groupby('Sender_account').agg(
    total_tx=('Amount', 'count'),
    is_laundering_sender=('Is_laundering', 'max')
).reset_index()

print(sender_counts.groupby('is_laundering_sender')['total_tx'].describe())

=== CLASS IMBALANCE METRICS ===
Is_laundering
Normal (0)        499444
Laundering (1)       556
Name: count, dtype: int64
Laundering Percentage: 0.1112%

=== MONEY LAUNDERING TYPOLOGY BREAKDOWN ===
     Laundering_type  Count
         Structuring    166
     Cash_Withdrawal     74
            Smurfing     59
   Stacked Bipartite     40
     Layered_Fan_Out     39
Behavioural_Change_1     38
        Deposit-Send     30
              Fan_In     17
Behavioural_Change_2     13
      Gather-Scatter     13
        Single_large     12
      Layered_Fan_In     12
               Cycle     12
      Scatter-Gather     11
           Bipartite     11
             Fan_Out      7
      Over-Invoicing      2

=== BEHAVIORAL FREQUENCY PER SENDER ACCOUNT ===
                        count       mean        std  min  25%  50%   75%  \
is_laundering_sender                                                       
0                     46104.0  10.773165  23.849369  1.0  1.0  8.0  12.0   
1                    

In [17]:
print("Calculating real-time behavioral velocities (handling simultaneous transactions)...")

df_features = df.copy()

# 1. Ensure absolute chronological order and maintain a unique row index (0 to N)
df_features = df_features.sort_values(by='Timestamp').reset_index(drop=True)

# 2. Compute rolling windows using 'on' to avoid duplicate index collisions
print("1/3 Computing 1-hour transaction count velocity...")
count_1h = df_features.groupby('Sender_account').rolling('1h', on='Timestamp')['Amount'].count()
# Drop the Sender_account from the groupby index, sort by the unique row index, and extract values
df_features['tx_count_1h'] = count_1h.reset_index(level=0, drop=True).sort_index().values

print("2/3 Computing 1-hour financial flow velocity...")
sum_1h = df_features.groupby('Sender_account').rolling('1h', on='Timestamp')['Amount'].sum()
df_features['tx_amount_sum_1h'] = sum_1h.reset_index(level=0, drop=True).sort_index().values

print("3/3 Computing 24-hour transaction count velocity...")
count_24h = df_features.groupby('Sender_account').rolling('24h', on='Timestamp')['Amount'].count()
df_features['tx_count_24h'] = count_24h.reset_index(level=0, drop=True).sort_index().values

# Robustly fill any NaNs created by rolling calculation limits
metrics = ['tx_count_1h', 'tx_amount_sum_1h', 'tx_count_24h']
df_features[metrics] = df_features[metrics].fillna(0)

print("\n=== FINAL VALIDATED FEATURES VS LAUNDERING FLAGS ===")
print(df_features.groupby('Is_laundering')[metrics].mean())

Calculating real-time behavioral velocities (handling simultaneous transactions)...
1/3 Computing 1-hour transaction count velocity...
2/3 Computing 1-hour financial flow velocity...
3/3 Computing 24-hour transaction count velocity...

=== FINAL VALIDATED FEATURES VS LAUNDERING FLAGS ===
               tx_count_1h  tx_amount_sum_1h  tx_count_24h
Is_laundering                                             
0                 3.615354      34876.579852     26.510337
1                 1.464029      29188.945368      5.715827
